In [1]:
import math
import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import torch

In [2]:
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.pipeline.data_handler import DataHandler
from src.utility import read_file, get_config, get_path
config = get_config.load()

In [4]:
SYMBOLS = [s.split('/')[0] for s in config['pipeline']['symbols']]
model_dir = get_path.absolute(config['path']['strategy']['model'])
portfolio_dir = get_path.absolute(config['path']['strategy']['portfolio'])
get_path.check(model_dir)
get_path.check(portfolio_dir)
seq_len = config['strategy']['sequence_length']
min_amt = config['backtest']['minimum_amount']
txn_cost = config['backtest']['transaction_cost_fraction']
capital = config['backtest']['capital']
bankruptcy_fraction = config['strategy']['bankruptcy_fraction']
slippage = config['strategy']['slippage_cost_fraction']
slm = config['strategy']['stop_loss_multiple']
slp = config['strategy']['stop_loss_fraction']
tpm = config['strategy']['take_profit_multiple']
tpp = config['strategy']['take_profit_fraction']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
data = read_file.read_data('train', 'data')
start_date = '2021-01-01'
end_date = '2022-01-01'

In [7]:
data_handler = DataHandler(data, SYMBOLS, start_date, end_date)

In [8]:
data_handler.get_latest_candle()

{}

In [11]:
for _ in range(4):
    data_handler.update_candles()

In [13]:
data_handler.get_latest_candle()
data_handler.get_latest_candle_datetime()

Timestamp('2021-01-01 04:00:00')

In [30]:
def display(df, symbols):
    # Print Header
    header = f"{'Timestamp':<20} | {'Symbol':<8} | {'Open':<10} | {'High':<10} | {'Low':<10} | {'Close':<10}"
    print(header)
    print("-" * len(header))

    # Iterate over rows using .iterrows()
    # `timestamp` is the DataFrame index, `row` contains the column values
    for timestamp, row in df.iterrows():
        ts_str = str(timestamp)

        for symbol in symbols:
            # Extract OHLC values using tuple keys for MultiIndex columns
            open_val  = row[('open', symbol)]
            high_val  = row[('high', symbol)]
            low_val   = row[('low', symbol)]
            close_val = row[('close', symbol)]

            print(f"{ts_str:<20} | {symbol:<8} | {open_val:<10.4f} | {high_val:<10.4f} | {low_val:<10.4f} | {close_val:<10.4f}")

In [31]:
data_handler.get_latest_candles(10)

,"(open, BNB)","(high, BNB)","(low, BNB)","(close, BNB)","(volume, BNB)","(macd-signal-pct, BNB)","(macd-slope, BNB)","(sar, BNB)","(tema, BNB)","(ppo, BNB)",...,"(adx, SOL)","(stoch-rsi, SOL)","(bop, SOL)","(natr, SOL)","(obv, SOL)","(vwap, SOL)","(mfi, SOL)","(log-return, SOL)","(vol-spike, SOL)","(norm-volatility, SOL)"
timestamp,,,,,,,,,,,,,,,,,,,,,
2021-01-01 00:00:00,37.3596,37.4423,36.9636,37.3764,95113.826,0.000505,0.019538,0.023028,0.004356,-0.595121,...,14.544020,1.851166e+01,0.783186,2.270607,-1952503.98,1.496985,42.786978,0.024186,-0.368500,0.029271
2021-01-01 01:00:00,37.3765,37.9390,37.3353,37.6134,152336.882,0.001149,0.035162,0.028691,0.006981,-0.478372,...,15.659307,1.404509e+01,0.375321,2.422485,-1695758.97,1.503304,55.840676,0.018733,1.500255,0.049447
2021-01-01 02:00:00,37.6134,37.9730,37.5758,37.9600,107655.644,0.002086,0.055746,0.036079,0.010179,-0.285798,...,16.694931,1.849595e+00,0.630542,2.321248,-1642480.70,1.505450,61.436166,0.008733,-0.464745,0.012790
2021-01-01 03:00:00,37.9600,38.1000,37.7551,37.9250,116381.591,0.002531,0.040796,0.033003,0.005109,-0.178129,...,17.986269,7.105427e-14,0.668085,2.239053,-1605512.70,1.508322,73.413302,0.009843,-0.571807,0.014661
2021-01-01 04:00:00,37.9240,38.0765,37.8300,37.8702,63709.814,0.002599,0.027039,0.028590,0.001177,-0.025394,...,19.240348,-4.522777e-01,-0.036458,2.165626,-1652114.87,1.510437,81.711767,-0.000437,-0.426411,0.011984


In [33]:
display(data_handler.get_latest_candles(10), SYMBOLS)

Timestamp            | Symbol   | Open       | High       | Low        | Close     
-----------------------------------------------------------------------------------
2021-01-01 00:00:00  | BNB      | 37.3596    | 37.4423    | 36.9636    | 37.3764   
2021-01-01 00:00:00  | BTC      | 28923.6300 | 29031.3400 | 28690.1700 | 28995.1300
2021-01-01 00:00:00  | ETH      | 736.4200   | 739.0000   | 729.3300   | 734.0700  
2021-01-01 00:00:00  | SOL      | 1.5088     | 1.5442     | 1.4990     | 1.5442    
2021-01-01 01:00:00  | BNB      | 37.3765    | 37.9390    | 37.3353    | 37.6134   
2021-01-01 01:00:00  | BTC      | 28995.1300 | 29470.0000 | 28960.3500 | 29409.9900
2021-01-01 01:00:00  | ETH      | 734.0800   | 749.0000   | 733.3700   | 748.2800  
2021-01-01 01:00:00  | SOL      | 1.5442     | 1.6100     | 1.5322     | 1.5734    
2021-01-01 02:00:00  | BNB      | 37.6134    | 37.9730    | 37.5758    | 37.9600   
2021-01-01 02:00:00  | BTC      | 29410.0000 | 29465.2600 | 29120.0300 | 291